# Multimodal notebook

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
!pip install unsloth
import unsloth
from unsloth import FastVisionModel
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive

/tmp/ipykernel_6744/3528354911.py:7: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_100.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SYSTEM_PROMPT_MULTIMODAL = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

In [ ]:
def load_model_unsloth(model_name, max_seq_length=4096, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    FastVisionModel.for_inference(model)

    return model, tokenizer

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    return data, ground_truth

In [ ]:
def filter_questions(data, ground_truth):
    """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']

            for q in exercise['questions']:
                q_id = q['questionId']

                if q_id in ground_truth:
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": q['options'],
                        "real": ground_truth[q_id]
                    })
    return tareas

In [ ]:
from PIL import Image

def prepare_batch(batch, system_prompt, modo_salida, ejemplos=None):
    """
    Construye los mensajes en formato multimodal para un lote de tareas.
    Ahora incluye ejemplos reales en el prompt del usuario (Few-Shot).
    """
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []

        # Añadir ejemplos al inicio del user_content si se proporcionan
        if ejemplos:
            for ej in ejemplos:
                example_text = f"--- EJEMPLO ----\nTexto:\n{ej['contexto']}\n\nPregunta: {ej['pregunta']}\nOpciones:\n"
                for opt in ej['opciones']:
                    letra = opt['optionId']
                    texto = opt.get('text', '').strip()
                    if texto:
                        example_text += f"{letra}) {texto}\n"
                example_text += f"\nRespuesta: {ej['real']}\n\n"
                user_content.append({"type": "text", "text": example_text})

        # Contenido de la tarea actual
        base_text_current_task = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n"
        user_content.append({"type": "text", "text": base_text_current_task})

        imagenes_tarea = []
        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})
            elif ruta_img:
                user_content.append({"type": "text", "text": f"{letra}) "})
                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)
                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        if modo_salida == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [ ]:
import torch

def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def show_results(stats, output_file):
    """Imprime por pantalla el resumen de la evaluación."""
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*50)
    print(f"RESULTADOS : {output_file}")
    print("="*50)
    if stats["errores_formato"] > 0:
        print(f"Errores de formato (JSON/Regex fallido): {stats['errores_formato']} de {stats['total']}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 50)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")

In [ ]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen


def run_inference(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size_texto=4, # Batch size solo para textos
    output_file="resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    todas_las_tareas = filter_questions(data, ground_truth)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todas_las_tareas)
    if os.path.exists(output_file):
        os.remove(output_file)

    ejemplos_para_prompt = []
    # Extraer el primer y último elemento de tareas_texto como ejemplos
    if len(tareas_texto) >= 2:
        ejemplos_para_prompt.append(tareas_texto[0])
        ejemplos_para_prompt.append(tareas_texto[-1])
        # Eliminar el primer y último elemento de la lista de tareas a procesar
        tareas_texto = tareas_texto[1:-1]
    elif len(tareas_texto) == 1:
        # Si solo hay una tarea, usarla como ejemplo y no procesar ninguna tarea de texto
        ejemplos_para_prompt.append(tareas_texto[0])
        tareas_texto = []
    # Si len(tareas_texto) es 0, ejemplos_para_prompt permanece vacío y tareas_texto también.


    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            # Pasar los ejemplos extraídos a prepare_batch
            mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida, ejemplos_para_prompt)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

                tarea_actual = batch[j]
                real = tarea_actual["real"]
                nivel = tarea_actual["nivel"]
                es_correcto = (prediccion == real)

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": nivel,
                    "pregunta": tarea_actual["pregunta"],
                    "respuesta_real": real,
                    "prediccion_modelo": prediccion,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                    "estado": "CORRECTO" if es_correcto else "INCORRECTO"
                })

            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    linea_json = json.dumps(resultado, ensure_ascii=False)
                    f.write(linea_json + '\n')

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} tareas de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} tareas MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

In [ ]:
def calculate_metrics(input_file="resultados.jsonl"):
    """Lee las predicciones almacenadas y calcula las métricas finales."""

    if not os.path.exists(input_file):
        print(f"Error: No se ha encontrado el archivo {input_file}.")
        return

    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    with open(input_file, 'r', encoding='utf-8') as f:
        for linea in f:
            if not linea.strip():
                continue

            resultado = json.loads(linea)

            nivel = resultado["nivel"]
            es_correcto = (resultado["estado"] == "CORRECTO")
            error_json = resultado.get("error_procesamiento_json", False)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            if error_json:
                stats["errores_formato"] += 1

    show_results(stats, input_file)

In [ ]:
def filter_and_calculate(input_file='prueba.json'):
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        subset_data = json.load(f)
        allowed_ids = set(subset_data.keys())

    output_path = 'filtered_results.jsonl'

    with open(gemma4_results_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        for linea in f_in:
            if not linea.strip():
                continue
            try:
                entry = json.loads(linea)
                if entry.get("questionId") in allowed_ids:
                    f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
            except json.JSONDecodeError:
                continue

    calculate_metrics(output_path)

In [ ]:
def preview_prompt(task, system_prompt, modo_salida, ejemplos=None):
    """
    Genera y muestra una vista previa del prompt para una única tarea.
    """
    print("--- PREVISUALIZACIÓN DEL PROMPT ---")

    # prepare_batch espera una lista de tareas, así que envolvemos la tarea única
    mensajes_batch, imagenes_batch = prepare_batch([task], system_prompt, modo_salida, ejemplos)

    # Extraemos el primer (y único) mensaje y las imágenes generadas
    mensajes = mensajes_batch[0]
    imagenes = imagenes_batch[0]

    for message in mensajes:
        print(f"\nROLE: {message['role'].upper()}")
        if isinstance(message['content'], list):
            for item in message['content']:
                if item['type'] == 'text':
                    print(item['text'])
                elif item['type'] == 'image':
                    print("[IMAGEN ADJUNTA]")
        else:
            print(message['content'])

    if imagenes:
        print(f"\n(NOTA: Esta tarea incluye {len(imagenes)} imagen(es) que serían procesadas por el modelo.)")
    print("----------------------------------")

### Previsualización del Prompt

In [ ]:
# Cargamos los datos para obtener una tarea de ejemplo y los ejemplos few-shot
data_full, ground_truth_full = load_data()
all_tareas = filter_questions(data_full, ground_truth_full)

# Seleccionamos una tarea de ejemplo (la tercera en este caso)
sample_task = all_tareas[2]

# Preparamos los ejemplos para el few-shot, como se hace en run_inference
tareas_texto_temp, _ = split_tasks_by_modality(all_tareas)
ejemplos_para_prompt = []
if len(tareas_texto_temp) >= 2:
    ejemplos_para_prompt.append(tareas_texto_temp[0])
    ejemplos_para_prompt.append(tareas_texto_temp[-1])
elif len(tareas_texto_temp) == 1:
    ejemplos_para_prompt.append(tareas_texto_temp[0])


# Llamamos a la función de previsualización
preview_prompt(
    task=sample_task,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL, # Usamos el system prompt multimodal
    modo_salida="letra",
    ejemplos=ejemplos_para_prompt
)

--- PREVISUALIZACIÓN DEL PROMPT ---

ROLE: SYSTEM
Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra.

ROLE: USER
--- EJEMPLO ----
Texto:
Duerida ALicia -
¿Qué tal estás? ¿Te gusta tu nuevo trabajo en Barcelona? Yo en Madrid
estoy muy bien. Tengo que darte buenas noticias. La próxima semana voy a
Barcelona con mi hija Celia, ¿ la recuerdas? Va a hacer este verano un curso
sobre cine allí. El curso es de un mes y empieza el 15 de agosto. Sólo tiene clases
por las mañanas, de nue

## [Gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it) cuantizado


In [ ]:
gemma4_model, gemma4_tokenizer = load_model_unsloth("unsloth/gemma-4-E4B-it-unsloth-bnb-4bit", max_seq_length=2048)

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results')

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_few_shot.json")

### zero-shot

In [ ]:
run_inference(
    model=gemma4_model,
    tokenizer=gemma4_tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size_texto=1,
    output_file=gemma4_results_path
)

calculate_metrics(input_file=gemma4_results_path)


--- Procesando 126 tareas de SOLO TEXTO (Batch Size: 1) ---


Progreso Texto: 100%|██████████| 126/126 [02:43<00:00,  1.29s/it]



--- Procesando 7 tareas MULTIMODALES (Batch Size: 1) ---


Progreso Imágenes: 100%|██████████| 7/7 [00:53<00:00,  7.63s/it]


Resultados guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_few_shot.json

RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_few_shot.json
Errores de formato (JSON/Regex fallido): 1 de 133
Accuracy Global: 83.46% (111/133)
--------------------------------------------------
Nivel A1: 87.14% (61/70)
Nivel A2: 78.26% (18/23)
Nivel B1: 76.19% (16/21)
Nivel B2: 84.21% (16/19)


In [ ]:
#@title Accuracy solo sobre ejemplos con imágenes
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_images.json")

filter_and_calculate(input_file=gemma4_results_path)


RESULTADOS : filtered_results.jsonl
Accuracy Global: 57.14% (4/7)
--------------------------------------------------
Nivel A1: 57.14% (4/7)
